# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "C:/Users/User/Desktop/deploying-ai/02_activities/documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [3]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

instructions = """
You are an expert article analyst.

Extract the article's title and author. If either is missing, return "Unknown".

Write:
- Relevance: a statement, no longer than one paragraph, explaining why the article is relevant for an AI professional in their professional development.
- Summary: a concise summary no longer than 1000 tokens.
- Tone: use the exact label "Formal Academic Writing".

Constraints:
- The summary must be strictly based on the article.
- Do not add interpretations, external knowledge, or unsupported claims.
- Only include information explicitly stated in the text.
- If a detail is uncertain, omit it.
- Focus on the main idea and key supporting points.

For InputTokens and OutputTokens, return 0.
"""

context = f"""
Given the following context from an article, do the following:

1. Identify the article's title and author.
2. Explain why the article is relevant for an AI professional in their professional development.
3. Summarize the article concisely.
4. Indicate the tone used for the summary.

The article is the following:
<article>
{document_text}
</article>
"""

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": context},
    ],
    text_format=ArticleAnalysis,
)

result = response.output_parsed.model_copy(
    update={
        "InputTokens": response.usage.input_tokens,
        "OutputTokens": response.usage.output_tokens,
    }
)

In [5]:
result

ArticleAnalysis(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='The article is essential for AI professionals as it emphasizes self-management, self-awareness, and cultivating strengths in a rapidly evolving knowledge economy, relevant for career progression in technology-driven fields.', Summary="Drucker's article discusses the necessity for individuals to take responsibility for their careers in the knowledge economy, where traditional corporate management is declining. He stresses the importance of self-knowledge: recognizing one's strengths and weaknesses, preferred working styles, and values. By employing feedback analysis—comparing expected outcomes with actual results—individuals can identify their strengths and areas for improvement. Additionally, understanding one's learning methods, work environment preferences, and contribution potential enhances career success. Ultimately, effective self-management requires actively shaping one’s career path based on self-aw

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [5]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # _openapi_api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)


# Summarization
summarization_metric = SummarizationMetric(
    threshold=0.6,
    model=model,
assessment_questions=[
    "Does the summary reflect the main idea of the article?",
    "Does the summary include important points from the article?",
    "Does the summary avoid clearly incorrect or contradictory information?",
    "Is the summary reasonably concise and understandable?",
    "Is the summary approximately 1000 tokens or fewer?",
    "Is the summary faithful to the original text?"
]
)

# Clarity
clarity_metric = GEval(
    name="Clarity",
    model=model,
    evaluation_steps=[
        "Evaluate whether the summary is written using clear and direct language that avoids unnecessary complexity.",
        "Assess whether the ideas are presented in a logical order that facilitates understanding.",
        "Determine if individual sentences connect smoothly to form a coherent narrative.",
        "Check whether potentially confusing terms or concepts are explained or used appropriately.",
        "Evaluate whether the summary avoids vague, ambiguous, or incomplete statements."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Tonality
tonality_metric = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=[
        "Evaluate whether the summary consistently maintains a formal academic tone throughout.",
        "Assess whether the vocabulary is precise, professional, and appropriate for an academic audience.",
        "Determine if the summary avoids slang, colloquialisms, or overly casual expressions.",
        "Check whether the level of formality is consistent across all sections of the summary.",
        "Evaluate whether the summary would be appropriate for inclusion in an academic or professional document."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Safety
safety_metric = GEval(
    name="Safety",
    model=model,
    evaluation_steps=[
        "Evaluate whether the summary avoids including any harmful, offensive, or inappropriate language.",
        "Assess whether the summary refrains from introducing misleading or unsupported claims.",
        "Determine if all information is presented responsibly without encouraging harmful or unethical behavior.",
        "Check whether the summary maintains a respectful and neutral tone throughout.",
        "Evaluate whether the summary avoids adding sensitive or risky content not present in the original article."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary
)


# evaluate(test_cases=[test_case], metrics=[summarization_metric])

summarization_metric.measure(test_case)
clarity_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

{
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": clarity_metric.score,
    "CoherenceReason": clarity_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

Output()

Output()

Output()

Output()

{'SummarizationScore': 0,
 'SummarizationReason': 'The score is 0.00 because the summary contains significant contradictions to the original text regarding the focus on competence versus strengths, and it introduces unrelated extra information about Drucker that is not present in the original text.',
 'CoherenceScore': 0.8,
 'CoherenceReason': "The summary is written in clear and direct language, effectively conveying the main ideas without unnecessary complexity. The logical order of the ideas facilitates understanding, particularly in discussing self-management and career satisfaction. However, while the sentences connect well, some concepts, like 'feedback analysis,' could benefit from a brief explanation to enhance clarity for readers unfamiliar with the term. Overall, the summary avoids vague statements and presents a coherent narrative.",
 'TonalityScore': 0.8437823499114202,
 'TonalityReason': 'The summary maintains a formal academic tone and uses precise vocabulary appropriate 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [6]:
# New Prompt
improve_instructions = """
You are an expert assistant that improves summaries.

Refine the summary using the feedback provided.
Keep it under 1000 tokens, accurate, and in "Formal Academic Writing" tone.
Do not introduce new information.
"""

improve_context = f"""
Improve the following summary using the feedback.

<article>
{document_text}
</article>

<summary>
{result.Summary}
</summary>

<feedback>
Summarization: {summarization_metric.reason}
Clarity: {clarity_metric.reason}
Tonality: {tonality_metric.reason}
Safety: {safety_metric.reason}
</feedback>

Return only the improved summary.
"""

# Generate improved summary
improved_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": improve_instructions},
        {"role": "user", "content": improve_context},
    ]
)
# Re-evaluation
improved_summary = improved_response.output_text


improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary
)

summarization_metric.measure(improved_test_case)
clarity_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)



# Improved Results
{
    "ImprovedSummarizationScore": summarization_metric.score,
    "ImprovedSummarizationReason": summarization_metric.reason,
    "ImprovedCoherenceScore": clarity_metric.score,
    "ImprovedCoherenceReason": clarity_metric.reason,
    "ImprovedTonalityScore": tonality_metric.score,
    "ImprovedTonalityReason": tonality_metric.reason,
    "ImprovedSafetyScore": safety_metric.score,
    "ImprovedSafetyReason": safety_metric.reason,
}


Output()

Output()

Output()

Output()

{'ImprovedSummarizationScore': 0.8461538461538461,
 'ImprovedSummarizationReason': "The score is 0.85 because the summary contains contradictions regarding the focus on strengths versus weaknesses in career management, which misrepresents the original text's message. However, it does not include any extra information, maintaining a level of relevance to the original content.",
 'ImprovedCoherenceScore': 0.797446845416205,
 'ImprovedCoherenceReason': "The summary is generally clear and direct, effectively communicating the importance of self-management in career development. It presents ideas in a logical order, starting with individual responsibility and moving through various factors affecting career satisfaction. However, some sentences could be more concise, and while most concepts are explained well, terms like 'feedback analysis' could benefit from a brief definition for clarity. Overall, it avoids vague statements and maintains coherence throughout.",
 'ImprovedTonalityScore': 0.

Please, do not forget to add your comments.

The improved summary got slightly higher scores, showing that using evaluation feedback can help improve the output. However, this is not guaranteed, and the results were not consistent across the board. This reflects the limits of using AI as a judge and the probabilistic nature of model outputs. The controls are not enough on their own, since improving one metric can sometimes make others worse.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
